# Report 3 — Diagnostic

**Question: does the null DANN result in Report 3 come from raw-input choice, from architecture, or from a DANN bug?**

Student: Sanjeev Veeramani (Matr. 100004303)
Program: M.Sc. Applied Data Science and Analytics, SRH University Heidelberg
Supervisor: Prof. Dr. Binh Vu

---

**Setup.** Report 3 trained CNN, LSTM, BiLSTM on raw 32×128 EEG segments with DANN on top. Result: DANN produced negative or negligible deltas in 8 of 9 (model × task) pairs. CNN was the most robust (max +0.76 pp); LSTM/BiLSTM were hurt more (up to −2.15 pp on BiLSTM valence).

**This diagnostic tests two hypotheses simultaneously.**

**H1 (input hypothesis):** DANN works, but raw EEG is the wrong input. Published DANN results on DEAP LOSO are generally obtained with **DE features**, that is 160-dim engineered per-channel per-band values, rather than raw waveforms.

**H2 (architecture hypothesis):** RNNs (LSTM, BiLSTM) have a specific interaction with DANN — the gradient reversal through backprop-through-time destabilises long-range representations more than it does convolutional features.

**Design.** Take the DE features already extracted in Report 2 Part 2 (`X_deap_DE_zscored.npy`, shape (152320, 160)). Feed them through two extractor variants: MLP (a standard DE-based setup) and a small LSTM (treats 160 dims as a sequence). Both with and without DANN. Valence only. 5 folds only.

**Reads.**
| Outcome | Interpretation |
|---|---|
| MLP + DANN ≫ MLP plain (≥ +5 pp) | H1 confirmed: DE input is the fix. R4 uses DE. |
| MLP + DANN ≈ MLP plain (< 2 pp) | DANN implementation itself is weak here — deeper investigation before R4. |
| MLP + DANN > LSTM + DANN by >5 pp | H2 confirmed: RNNs have specific DANN problem. R4 architecture strategy changes. |
| Both ≈ 53% | Report 3 result is not an input or architecture issue — genuinely hard task. |

**Estimated runtime:** ~10 minutes on L4 GPU (4 configurations × 5 folds × short training).

## 1. Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/case_study_2'
assert os.path.isdir(PROJECT_ROOT), f'Set PROJECT_ROOT correctly. Got {PROJECT_ROOT}'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import torch, numpy as np, pandas as pd, time, gc, pickle
from pathlib import Path
from tqdm.auto import tqdm
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from torch.autograd import Function
from sklearn.metrics import accuracy_score, balanced_accuracy_score

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Device: NVIDIA L4


## 2. Configuration
One task (valence), 5 folds, two architectures, two variants = 4 runs × 5 folds = 20 fold-trainings.

In [4]:
CONFIG = {
    'data_path':       'processed/X_deap_DE_zscored.npy',   # (N, 160) — DE per-subject z-scored
    'subject_id_path': 'processed/deap_subject_ids.npy',
    'valence_path':    'processed/y_deap_valence.npy',

    'archs':    ['MLP', 'LSTM'],     # MLP for input hypothesis, LSTM for arch hypothesis
    'variants': ['plain', 'dann'],
    'folds_to_run': [1, 2, 3, 4, 5],  # 5-fold smoke test

    'batch_size':    512,
    'epochs':        15,
    'lr':            1e-3,
    'weight_decay':  1e-4,
    'dann_lambda_max':  1.0,
    'dann_ramp_epochs': 8,
}

## 3. Load DE features
Expected shape: (152320, 160). If it's already flat per-segment, use directly. If per-trial, reshape.

In [5]:
root = Path(PROJECT_ROOT)
X = np.load(root / CONFIG['data_path']).astype(np.float32)
SUBJ = np.load(root / CONFIG['subject_id_path'])
Y = np.load(root / CONFIG['valence_path']).astype(np.int64)

print(f'X: {X.shape}, dtype={X.dtype}')
print(f'SUBJ: {SUBJ.shape}, unique={np.unique(SUBJ).tolist()}')
print(f'Y (valence): {Y.shape}, bincount={np.bincount(Y)}')

assert X.shape[0] == SUBJ.shape[0] == Y.shape[0], 'Length mismatch'
assert X.ndim == 2 and X.shape[1] == 160, f'Expected (N, 160), got {X.shape}'

X: (152320, 160), dtype=float32
SUBJ: (152320,), unique=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
Y (valence): (152320,), bincount=[68068 84252]


## 4. Models

**MLP extractor** — matches how Li et al. 2018 (DANN 69.20%) uses DE features. Three linear layers with dropout.

**LSTM extractor** — treats the 160-dim DE vector as a sequence of 32 timesteps × 5 features (5 frequency bands per channel). Tests whether the RNN inductive bias helps on engineered features.

In [6]:
class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None

def grad_reverse(x, alpha=1.0):
    return GradientReversalFunction.apply(x, alpha)

EMBED_DIM = 128

class MLPExtractor(nn.Module):
    def __init__(self, in_dim=160, embed=EMBED_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128),    nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, embed),  nn.ReLU(), nn.Dropout(0.3),
        )
    def forward(self, x):
        return self.net(x)

class LSTMExtractor(nn.Module):
    def __init__(self, in_feat=5, hidden=64, embed=EMBED_DIM):
        # Reshape (N, 160) -> (N, 32, 5): 32 channels as timesteps, 5 bands as features
        super().__init__()
        self.lstm = nn.LSTM(in_feat, hidden, num_layers=2, batch_first=True, dropout=0.3)
        self.fc = nn.Sequential(nn.Linear(hidden, embed), nn.ReLU(), nn.Dropout(0.3))
    def forward(self, x):
        x = x.view(-1, 32, 5)         # (N, 32, 5)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

def make_extractor(name):
    if name == 'MLP':  return MLPExtractor()
    if name == 'LSTM': return LSTMExtractor()
    raise ValueError(name)

class DiagModel(nn.Module):
    def __init__(self, arch, n_classes=2, use_dann=False, n_domains=None):
        super().__init__()
        self.extractor  = make_extractor(arch)
        self.label_head = nn.Sequential(
            nn.Linear(EMBED_DIM, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, n_classes),
        )
        self.use_dann = use_dann
        if use_dann:
            self.domain_head = nn.Sequential(
                nn.Linear(EMBED_DIM, 64), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(64, n_domains),
            )
    def forward(self, x, alpha=0.0):
        feat = self.extractor(x)
        logits_label = self.label_head(feat)
        if self.use_dann:
            feat_rev = grad_reverse(feat, alpha)
            return logits_label, self.domain_head(feat_rev)
        return logits_label, None

## 5. Training utilities

In [7]:
def make_sampler(y):
    counts = np.bincount(y)
    weights = 1.0 / counts[y]
    return WeightedRandomSampler(torch.DoubleTensor(weights), len(y), replacement=True)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    for batch in loader:
        xb = batch[0].to(DEVICE, non_blocking=True)
        yb = batch[1]
        logits, _ = model(xb, alpha=0.0)
        ps.append(logits.argmax(1).cpu().numpy())
        ys.append(yb.numpy())
    y = np.concatenate(ys); p = np.concatenate(ps)
    return y, p, balanced_accuracy_score(y, p)

def train_fold(model, train_loader, test_loader, use_dann, epochs, lr, wd,
               dann_lambda_max, dann_ramp_epochs, log_domain_acc=True):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss()

    best_bal, best_state, best_ep = -1.0, None, 0
    domain_acc_final = None

    for ep in range(epochs):
        alpha = dann_lambda_max * min(1.0, ep / max(dann_ramp_epochs, 1)) if use_dann else 0.0
        model.train()
        domain_correct, domain_total = 0, 0
        for xb, yb, sb in train_loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE); sb = sb.to(DEVICE)
            opt.zero_grad()
            logits, dom_logits = model(xb, alpha=alpha)
            loss = loss_fn(logits, yb)
            if use_dann and dom_logits is not None:
                loss = loss + loss_fn(dom_logits, sb)
                domain_correct += (dom_logits.argmax(1) == sb).sum().item()
                domain_total += sb.size(0)
            loss.backward()
            opt.step()
        sched.step()

        _, _, bal = evaluate(model, test_loader)
        if bal > best_bal:
            best_bal, best_ep = bal, ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if use_dann and log_domain_acc and ep == epochs - 1:
            domain_acc_final = 100 * domain_correct / max(domain_total, 1)

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_bal, best_ep, domain_acc_final

## 6. Diagnostic runner
Runs 4 configurations (2 archs × 2 variants) × 5 folds each. Logs balanced accuracy AND final domain-classifier accuracy for DANN runs — the domain-acc tells us if the gradient reversal is doing anything at all.

In [8]:
def loso_split(subj, held_out):
    tr = np.where(subj != held_out)[0]; te = np.where(subj == held_out)[0]
    return tr, te

def run_config(arch, variant):
    use_dann = (variant == 'dann')
    print(f'\n=== {arch} | valence | {variant} ===')
    per_fold = []
    for held_out in CONFIG['folds_to_run']:
        tr_idx, te_idx = loso_split(SUBJ, held_out)
        X_tr, y_tr, s_tr = X[tr_idx], Y[tr_idx], SUBJ[tr_idx]
        X_te, y_te       = X[te_idx], Y[te_idx]

        s_tr_remap = np.searchsorted(np.unique(s_tr), s_tr)
        n_domains = int(s_tr_remap.max()) + 1

        train_ds = TensorDataset(torch.from_numpy(X_tr),
                                 torch.from_numpy(y_tr).long(),
                                 torch.from_numpy(s_tr_remap).long())
        test_ds  = TensorDataset(torch.from_numpy(X_te),
                                 torch.from_numpy(y_te).long())
        train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'],
                                  sampler=make_sampler(y_tr), num_workers=2, pin_memory=True, drop_last=True)
        test_loader  = DataLoader(test_ds, batch_size=CONFIG['batch_size']*2, shuffle=False,
                                  num_workers=2, pin_memory=True)

        model = DiagModel(arch, n_classes=2, use_dann=use_dann, n_domains=n_domains)

        t0 = time.time()
        model, best_bal, best_ep, dom_acc = train_fold(
            model, train_loader, test_loader, use_dann,
            CONFIG['epochs'], CONFIG['lr'], CONFIG['weight_decay'],
            CONFIG['dann_lambda_max'], CONFIG['dann_ramp_epochs'])
        elapsed = time.time() - t0

        y_true, y_pred, bal = evaluate(model, test_loader)
        acc = accuracy_score(y_true, y_pred)
        row = {'arch': arch, 'variant': variant, 'sid': int(held_out),
               'acc': 100*acc, 'bal': 100*bal, 'best_ep': best_ep,
               'domain_acc': dom_acc if dom_acc is not None else np.nan,
               'time_s': elapsed}
        per_fold.append(row)
        print(f'  S{held_out}: bal={100*bal:.2f}%  best_ep={best_ep}  dom_acc={dom_acc}  time={elapsed:.1f}s')

        del model, train_loader, test_loader
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    return per_fold

In [9]:
ALL = []
for arch in CONFIG['archs']:
    for variant in CONFIG['variants']:
        ALL.extend(run_config(arch, variant))

results = pd.DataFrame(ALL)
print('\nDone.')


=== MLP | valence | plain ===
  S1: bal=47.60%  best_ep=0  dom_acc=None  time=28.9s
  S2: bal=53.66%  best_ep=5  dom_acc=None  time=18.2s
  S3: bal=55.83%  best_ep=2  dom_acc=None  time=18.4s
  S4: bal=53.35%  best_ep=5  dom_acc=None  time=17.9s
  S5: bal=58.24%  best_ep=9  dom_acc=None  time=18.0s

=== MLP | valence | dann ===
  S1: bal=48.50%  best_ep=8  dom_acc=3.805881076388889  time=21.8s
  S2: bal=56.72%  best_ep=3  dom_acc=3.7841796875  time=21.7s
  S3: bal=56.20%  best_ep=0  dom_acc=3.750271267361111  time=21.7s
  S4: bal=53.28%  best_ep=0  dom_acc=3.7129720052083335  time=21.7s
  S5: bal=57.46%  best_ep=7  dom_acc=3.774685329861111  time=21.7s

=== LSTM | valence | plain ===
  S1: bal=51.05%  best_ep=14  dom_acc=None  time=22.3s
  S2: bal=46.10%  best_ep=6  dom_acc=None  time=21.7s
  S3: bal=56.06%  best_ep=0  dom_acc=None  time=21.9s
  S4: bal=42.61%  best_ep=4  dom_acc=None  time=21.6s
  S5: bal=59.38%  best_ep=0  dom_acc=None  time=21.7s

=== LSTM | valence | dann ===
  S1

## 7. Results and interpretation

In [10]:
summary = results.groupby(['arch', 'variant']).agg(
    bal_mean = ('bal', 'mean'),
    bal_std  = ('bal', 'std'),
    domain_acc_mean = ('domain_acc', 'mean'),
).reset_index()
print(summary.to_string(index=False))
print()

# Delta plain vs dann
pivot = summary.pivot(index='arch', columns='variant', values='bal_mean')
pivot['delta_pp'] = pivot['dann'] - pivot['plain']
print('Plain vs DANN deltas:')
print(pivot.to_string())

arch variant  bal_mean  bal_std  domain_acc_mean
LSTM    dann 52.002178 6.033990         3.360460
LSTM   plain 51.040078 6.893373              NaN
 MLP    dann 54.431817 3.677599         3.765598
 MLP   plain 53.734226 3.952420              NaN

Plain vs DANN deltas:
variant       dann      plain  delta_pp
arch                                   
LSTM     52.002178  51.040078  0.962101
MLP      54.431817  53.734226  0.697591


## 8. Reading the output

Compare against the raw-input Report 3 numbers:

| Config | Raw input (R3) | DE input (this diagnostic) |
|---|---|---|
| CNN plain valence | 53.80% | — |
| CNN DANN valence | 53.70% | MLP DANN result |
| LSTM plain valence | 52.87% | — |
| LSTM DANN valence | 52.10% | LSTM DANN result |

**Read as follows:**

- **If MLP + DANN reaches 60–70%:** H1 confirmed. DE features fix DANN. Report 4 uses DE across all models.
- **If MLP + DANN stays around 53%:** DANN implementation itself is the weak link. Investigate α schedule, domain-head capacity, or reference implementation before Report 4.
- **If MLP + DANN improves but LSTM + DANN doesn't:** H2 confirmed. RNNs have a specific DANN interaction. R4 uses attention over convolutional features rather than RNN backbones.
- **If domain_acc_mean is near 100/n_domains (i.e. random):** the gradient reversal is working — extractor is producing subject-invariant features.
- **If domain_acc_mean is >30%:** the gradient reversal isn't strong enough to force invariance — α is too low.

The output from Section 7 answers all four questions from one run.